# Lab 5: Clean the bike rentals

In [1]:
import os
import pandas as pd
import numpy as np

rentals_raw = pd.read_csv(r'C:\Users\Hey HP\Desktop\gowthami_py\Data Visualisation and Analysis\bike_rentals.csv')
rentals = rentals_raw.copy()

# Part 1: Practise

In [4]:
# Task 1: Load it, and remove the empty columns
print(rentals.shape)         
print(rentals.dtypes)
# price has the type str (text), so it is not a number yet.

empty = rentals.columns[rentals.isna().all()]
print(list(empty))            # coupon_code, damage_notes, late_fee_note
rentals = rentals.dropna(axis=1, how='all')
print(rentals.shape)                          
print(rentals.isna().all().sum())             

(403, 12)
rental_id           str
station             str
bike_type           str
helmets_text        str
is_member           str
insured             str
paid_online         str
status              str
start_date          str
end_date            str
price               str
rating          float64
dtype: object
[]
(403, 12)
0


In [6]:
# Task 2: The price, from text to numbers
# Step 1: remove the currency in front. Step 2: remove the comma for thousands. Then float.
p = rentals_raw['price'].str.replace('SEK ', '', regex=False)
print(p.head(3).tolist())        
p = p.str.replace(',', '', regex=False)
print(p.head(3).tolist())        

rentals['price'] = p.astype(float)
print(rentals['price'].dtype)                 # float64
rentals['price'].describe().round(1)


['855.50', '1,311.50', '350.00']
['855.50', '1311.50', '350.00']
float64


count     352.0
mean      847.3
std       633.4
min       154.5
25%       406.0
50%       668.2
75%      1194.6
max      3401.0
Name: price, dtype: float64

In [7]:
rentals.sort_values('price', ascending=False)[['rental_id', 'bike_type', 'status', 'price']].head(3)
# R-5282 (Cargo bike, 3401.0), R-5209 (Cargo bike, 2979.5), R-5378 (E-bike, 2941.0)

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


In [8]:
# Task 3: The missing prices
price_missing = rentals['price'].isna()
print(price_missing.sum(), round(price_missing.mean() * 100, 1))    # 51, 12.7 %

cancelled = rentals['status'] == 'Cancelled'
pd.crosstab(price_missing, cancelled)
# 47 of the 51 missing prices are cancelled rentals: nothing was paid, so no price. Missing FOR A REASON.
# 4 are not cancelled: missing BY ACCIDENT.
# Decision: I leave all 51 as missing. I do not invent prices. Every price statistic is based on
# 352 rentals, not 403, and I write that number next to the result.

51 12.7


status,False,True
price,,
False,352,0
True,4,47


In [9]:
# Task 4: The letters t and f
tf_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
print(tf_cols)                  # ['is_member', 'insured', 'paid_online']
# status is not in the list: it has three different texts (Completed, Cancelled, No-show), not only t and f.

for c in tf_cols:
    rentals[c] = rentals_raw[c].map({'t': True, 'f': False})

print(rentals[tf_cols].isna().sum())          # insured 3, the others 0
print(rentals['is_member'].value_counts())    # False 288, True 115

['is_member', 'insured', 'paid_online']
is_member      0
insured        3
paid_online    0
dtype: int64
is_member
False    288
True     115
Name: count, dtype: int64


In [10]:
rentals.groupby('is_member')['price'].agg(['count', 'median'])
# Non-members: median 756 (248 rentals with a price). Members: median 558 (104 rentals).
# Members pay less in this table. But I cannot say membership is the reason: they might rent
# cheaper bikes or fewer days. I would need to compare like with like.

,count,median
is_member,,
False,248,756.25
True,104,558.00


In [11]:
# Task 5: The dates  
print(rentals_raw[['start_date', 'end_date']].isna().sum())    # 0 and 0 missing before

rentals['start_date'] = pd.to_datetime(rentals_raw['start_date'], errors='coerce')
rentals['end_date'] = pd.to_datetime(rentals_raw['end_date'], errors='coerce')
print(rentals[['start_date', 'end_date']].isna().sum())        # start_date 2, end_date 0
print(rentals[['start_date', 'end_date']].dtypes)              # datetime64[us]

# The 2 new missing values are impossible dates like 30 February / 31 June. errors='coerce' turned them into NaT.
print(rentals_raw.loc[rentals['start_date'].isna(), ['rental_id', 'start_date']])

start_date    0
end_date      0
dtype: int64
start_date    2
end_date      0
dtype: int64
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object
    rental_id  start_date
58     R-5335  2026-02-30
292    R-5063  2026-06-31


In [12]:
rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
rentals['rental_days'].describe().round(2)     # count 401, mean 2.96, min -2, max 7
rentals[rentals['rental_days'] < 0][['rental_id', 'start_date', 'end_date', 'rental_days']]
# The minimum is -2: a rental cannot end before it starts. R-5077 and R-5217 end before they start.
# I do not fix it in Part 1.

,rental_id,start_date,end_date,rental_days
6,R-5077,2026-07-19,2026-07-18,-1.0
196,R-5217,2026-06-20,2026-06-18,-2.0


In [13]:
# Task 6: The missing ratings
print(rentals['rating'].isna().sum())                           # 115
pd.crosstab(rentals['rating'].isna(), rentals['status'])
# 47 cancelled, 22 no-shows and 46 COMPLETED. Not a perfect match like in the lecture:
# a completed rental can be without a rating because the customer did not give one.

rentals['has_rating'] = rentals['rating'].notna()
print(rentals['has_rating'].value_counts())     # True 288, False 115
rentals['rating'].describe().round(2)           # median 5.0, mean 4.23

115
has_rating
True     288
False    115
Name: count, dtype: int64


count    288.00
mean       4.23
std        1.02
min        1.00
25%        4.00
50%        5.00
75%        5.00
max        5.00
Name: rating, dtype: float64

In [14]:
# Task 7: The helmets text
text = rentals['helmets_text']
print(text.value_counts(dropna=False))
# 1 helmet 146, 2 helmets 127, No helmets 59, 1 child helmet 47, 3 helmets 20, missing 4

rentals['child_helmet'] = text.str.contains('child')
print(rentals['child_helmet'].value_counts(dropna=False))    # True 47

helmets_text
1 helmet          146
2 helmets         127
No helmets         59
1 child helmet     47
3 helmets          20
NaN                 4
Name: count, dtype: int64
child_helmet
False    356
True      47
Name: count, dtype: int64


In [ ]:
# extract takes the first number from the text. 'No helmets' has no number, so it gets NaN.
count = pd.to_numeric(text.str.extract(r'(\d+)', expand=False))
# where keeps the number where the condition is True, else 0. The condition: NOT 'no helmets'.
no_helmets = text.str.lower().str.contains('no helmets', na=False)
rentals['helmets_count'] = count.where(~no_helmets, 0)
rentals['helmets_count'].value_counts(dropna=False).sort_index()
# 0 -> 59, 1 -> 193, 2 -> 127, 3 -> 20, missing 4 (the 4 missing texts stay missing)

In [15]:
# Task 8: Spellings and duplicates
for c in ['station', 'bike_type', 'status']:
    print(c, rentals_raw[c].nunique(), '->', rentals_raw[c].str.strip().str.lower().nunique())
# station 20 -> 5 (the problem column), bike_type 3 -> 3, status 3 -> 3

station 20 -> 5
bike_type 3 -> 3
status 3 -> 3


In [16]:
rentals['station'] = rentals_raw['station'].str.strip().str.title()
print(rentals['station'].value_counts())
# Central 107, Södermalm 95, Kungsholmen 89, Östermalm 57, Hammarby 55

station
Central        107
Södermalm       95
Kungsholmen     89
Östermalm       57
Hammarby        55
Name: count, dtype: int64


In [17]:
print('Duplicate rows:', rentals.duplicated().sum())                    # 3
print('Duplicate rental_id:', rentals.duplicated(subset='rental_id').sum())  # 3
rentals = rentals.drop_duplicates().reset_index(drop=True)
print(rentals.shape)                                                 # (400, 16)

Duplicate rows: 3
Duplicate rental_id: 3
(400, 15)


# Part 2: Go deeper

In [18]:
# Task 1: Which station is the most expensive?
# (a) all rentals with a price
a = (rentals.groupby('station')
            .agg(n=('price', 'count'), median=('price', 'median'))
            .sort_values('median', ascending=False))
a

,n,median
station,,
Kungsholmen,73,742.0
Södermalm,87,720.0
Östermalm,48,709.5
Hammarby,47,609.0
Central,95,596.0


In [19]:
# (b) only e-bikes
ebikes = rentals[rentals['bike_type'] == 'E-bike']
b = (ebikes.groupby('station')
           .agg(n=('price', 'count'), median=('price', 'median'))
           .sort_values('median', ascending=False))
b

,n,median
station,,
Kungsholmen,22,1554.0
Södermalm,25,1220.0
Central,27,1143.5
Östermalm,19,1087.5
Hammarby,15,711.5


In [20]:
# (c) share of e-bikes per station (mean of True/False times 100)
c = ((rentals['bike_type'] == 'E-bike').groupby(rentals['station']).mean() * 100).round(0)
c.sort_values(ascending=False)
# (a) Kungsholmen 73 rentals, median 742 is first. (b) Kungsholmen is still first (22 rentals, median 1554).
# Central goes from last to third. The e-bike share is about the same everywhere (28-39 %),
# so the mix of bikes does not explain the order.
# I can say: "Kungsholmen has the highest median price, also for e-bikes."
# I cannot say: "Kungsholmen is the most expensive station", because only 15 to 27 e-bike rentals
# stand behind each number, and the medians are close.

station
Östermalm      39.0
Södermalm      31.0
Hammarby       31.0
Kungsholmen    30.0
Central        28.0
Name: bike_type, dtype: float64

In [21]:
# Task 2: Values that cannot be true
rentals[rentals['rental_days'] < 0][['rental_id', 'start_date', 'end_date', 'rental_days']]   # R-5077 (-1), R-5217 (-2)

mean_before = rentals['rental_days'].mean()
rentals['rental_days'] = rentals['rental_days'].where(rentals['rental_days'] >= 0)
print(rentals['rental_days'].isna().sum())                       # 4
print(round(mean_before, 2), '->', round(rentals['rental_days'].mean(), 2))   # 2.96 -> 2.98
# 4 and not 2: the 2 negative values plus the 2 rentals whose start date was impossible (already missing).
# I would ask the shop: were the start and end dates typed in the wrong order?

4
2.96 -> 2.98


In [22]:
# Task 3: Everything in one cell, and save
# One cell: starts from rentals_raw every time, so it can be run again and again.
rentals = rentals_raw.copy()

# 1. empty columns
rentals = rentals.dropna(axis=1, how='all')

# 2. price: remove 'SEK ' and the comma, then float
rentals['price'] = (rentals['price'].str.replace('SEK ', '', regex=False)
                    .str.replace(',', '', regex=False).astype(float))

# 3. t and f -> True / False
tf_cols = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
for c in tf_cols:
    rentals[c] = rentals[c].map({'t': True, 'f': False})

# 4. dates (impossible dates become NaT) and rental_days (negative days become missing)
rentals['start_date'] = pd.to_datetime(rentals['start_date'], errors='coerce')
rentals['end_date'] = pd.to_datetime(rentals['end_date'], errors='coerce')
rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
rentals['rental_days'] = rentals['rental_days'].where(rentals['rental_days'] >= 0)

# 5. ratings and helmets
rentals['has_rating'] = rentals['rating'].notna()
text = rentals['helmets_text']
rentals['child_helmet'] = text.str.contains('child')
count = pd.to_numeric(text.str.extract(r'(\d+)', expand=False))
rentals['helmets_count'] = count.where(~text.str.lower().str.contains('no helmets', na=False), 0)

# 6. spelling of station, then duplicates last
rentals['station'] = rentals['station'].str.strip().str.title()
rentals_clean = rentals.drop_duplicates().reset_index(drop=True)

print(rentals_clean.shape)                                   # (400, 16)

os.makedirs('data', exist_ok=True)                           # create the data folder if it is missing
rentals_clean.to_csv('data/bike_rentals_clean.csv', index=False)

(400, 16)


In [23]:
back = pd.read_csv('data/bike_rentals_clean.csv', parse_dates=['start_date', 'end_date'])
print(back.shape)                                            # (400, 16)
print(back[['price', 'is_member', 'insured', 'start_date']].dtypes)
# price float64, is_member bool, start_date datetime64[us] (only because of parse_dates).
# NOT what I expected: insured is object, not bool. It has 3 missing values, and a CSV file cannot
# keep True / False / missing as a bool type, so pandas reads it as object.
# A CSV file only stores text: types are guessed again when the file is read.

(400, 16)
price                float64
is_member               bool
insured               object
start_date    datetime64[us]
dtype: object


## Task 4: My decisions for the bike rentals

I found nine problems in the table. For each one, here is what I did, why, and what it does to the numbers.

**1. Empty columns (3: `coupon_code`, `damage_notes`, `late_fee_note`)**
Removed. Every value in them is missing, so they carry no information. Nothing is lost. The table goes from 15 to 12 columns.

**2. Price as text**
Changed to a number. The price was text like `SEK 1,311.50`, so I could not sort or calculate with it. I removed `SEK ` and the comma in two steps, then changed it to float. Now `describe()` works: 352 prices, mean 847.3, median 668.2, max 3401.0.

**3. Missing prices (51, 12.7 %)**
Left as missing. 47 of them are cancelled rentals, where nothing was paid, so the price is missing for a reason. Only 4 are missing by accident. I do not want to invent prices, so every price result is based on 352 rentals, and I write that number next to the result.

**4. Impossible dates (2)**
Turned into missing (NaT). Two start dates, like 31 June and 30 February, do not exist. I used `errors='coerce'`. `start_date` now has 2 missing values.

**5. Rentals that end before they start (2: R-5077 and R-5217)**
Turned the negative `rental_days` into missing. A rental cannot have -1 or -2 days. I do not know which date is wrong, so I do not guess. In total 4 `rental_days` are missing (these 2 and the 2 from the impossible start dates). The mean goes from 2.96 to 2.98. I would ask the shop whether the dates were typed in the wrong order.

**6. Missing ratings (115)**
Left as missing, and I added a column `has_rating`. 47 are cancelled and 22 are no-shows, which makes sense. 46 are completed rentals where the customer did not leave a rating. I cannot know what they would have said, so I do not fill them. The median rating is 5.0 (288 ratings).

**7. The helmets text**
Changed into two columns. `child_helmet` is True if the text has the word child (47 rentals). `helmets_count` is the number from the text, and "No helmets" is 0 (59 rentals). The 4 rentals with no text stay missing.

**8. Spellings of `station`**
Fixed with `strip` and `title`. There were 20 different spellings (`CENTRAL`, ` Central`, `södermalm` and so on), and only 5 real stations: Central 107, Södermalm 95, Kungsholmen 89, Östermalm 57, Hammarby 55 (before the duplicates were removed).

**9. Duplicates (3)**
Removed 3 complete duplicate rows (the same 3 rental_id). The table goes from 403 to 400 rows.

Also, `insured` has 3 missing values. I left them as missing.

## Task 5: Your own question

In [24]:
# Question: Which bike type is cancelled most often?
by_type = (rentals_clean.assign(cancelled=rentals_clean['status'] == 'Cancelled')
                        .groupby('bike_type')
                        .agg(n_rentals=('rental_id', 'count'), share_cancelled=('cancelled', 'mean'))
                        .round(3)
                        .sort_values('share_cancelled', ascending=False))
by_type
# This does NOT tell me WHY rentals are cancelled. A cancelled rental may have been cancelled by the
# customer or by the shop (for example no bike available). There are only 44 cargo bike rentals,
# so a difference can be chance. And I don't know when the cancellation happened.
# (Replace this with a question of your own!)

,n_rentals,share_cancelled
bike_type,,
E-bike,124,0.121
City bike,232,0.116
Cargo bike,44,0.091
